# 检查 responses.csv 的 content 是否为有意义中文

规则：`content` 去除空白和标点符号后，必须至少包含 1 个中文汉字（`\u4e00-\u9fff`）。

输出：
- 不干净回答数量
- 不干净回答的 `participant_id`、`item_name`、`response_index`、`content`

In [ ]:
import re
import unicodedata
from pathlib import Path

import pandas as pd


# 笔记本位于 test/，数据位于 ../data/preprocessed/responses.csv
csv_path = Path("../data/preprocessed/responses.csv")
df = pd.read_csv(csv_path)

required_cols = ["participant_id", "item_name", "response_index", "content"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"responses.csv 缺少必要列: {missing_cols}")


def has_meaningful_chinese(text: object) -> bool:
    """去除空白/标点/符号后，至少包含一个中文汉字，才视为有意义中文。"""
    if pd.isna(text):
        return False

    s = str(text).strip()
    if not s:
        return False

    cleaned = "".join(
        ch
        for ch in s
        if not ch.isspace() and not unicodedata.category(ch).startswith(("P", "S"))
    )

    return re.search(r"[\u4e00-\u9fff]", cleaned) is not None


clean_mask = df["content"].apply(has_meaningful_chinese)
dirty_df = df.loc[
    ~clean_mask,
    ["participant_id", "item_name", "response_index", "content"],
].reset_index(drop=True)

print(f"不干净回答数量: {len(dirty_df)}")
display(dirty_df)